In [0]:
# Завдання 1. Читання, вибір колонок і фільтрація

from pyspark.sql import functions as F

trips_clean = spark.table("samples.nyctaxi.trips")

trips_clean = (
    trips_clean
    .select(
        "tpep_pickup_datetime", 
        "fare_amount", 
        "trip_distance", 
        "pickup_zip", 
        "dropoff_zip"
    )
    .filter(
        (F.col("pickup_zip").isNotNull()) 
        & (F.col("dropoff_zip").isNotNull())
        & (F.col("trip_distance") > 0)
        & (F.col("fare_amount") > 0)
    )   
)
    
display(trips_clean.limit(5))

tpep_pickup_datetime,fare_amount,trip_distance,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,8.0,1.4,10103,10110
2016-02-13T18:29:09.000Z,7.5,1.31,10023,10023
2016-02-06T19:40:58.000Z,9.5,1.8,10001,10018
2016-02-12T19:06:43.000Z,11.5,2.3,10044,10111
2016-02-23T10:27:56.000Z,18.5,2.6,10199,10022


In [0]:
# Завдання 2. Нові колонки

trips_prepared = (
    trips_clean
    .withColumn("pickup_month", F.month(F.col("tpep_pickup_datetime")))
    .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2))

)
display(trips_prepared.limit(5))

tpep_pickup_datetime,fare_amount,trip_distance,pickup_zip,dropoff_zip,pickup_month,fare_per_mile
2016-02-13T21:47:53.000Z,8.0,1.4,10103,10110,2,5.71
2016-02-13T18:29:09.000Z,7.5,1.31,10023,10023,2,5.73
2016-02-06T19:40:58.000Z,9.5,1.8,10001,10018,2,5.28
2016-02-12T19:06:43.000Z,11.5,2.3,10044,10111,2,5.0
2016-02-23T10:27:56.000Z,18.5,2.6,10199,10022,2,7.12


In [0]:
# Завдання 3. Запис у файли тома і читання файлів

spark.sql("""CREATE VOLUME IF NOT EXISTS workspace.default.taxi_files""")

output_path = "/Volumes/workspace/default/taxi_files/trips_prepared"

(
    trips_prepared.write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(output_path)
)


trips_from_files = spark.read.parquet(output_path)

display(trips_from_files.limit(5))

display(dbutils.fs.ls(output_path))
trips_from_files.count()



tpep_pickup_datetime,fare_amount,trip_distance,pickup_zip,dropoff_zip,fare_per_mile,pickup_month
2016-02-13T21:47:53.000Z,8.0,1.4,10103,10110,5.71,2
2016-02-13T18:29:09.000Z,7.5,1.31,10023,10023,5.73,2
2016-02-06T19:40:58.000Z,9.5,1.8,10001,10018,5.28,2
2016-02-12T19:06:43.000Z,11.5,2.3,10044,10111,5.0,2
2016-02-23T10:27:56.000Z,18.5,2.6,10199,10022,7.12,2


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/_SUCCESS,_SUCCESS,0,1792684770000
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=1/,pickup_month=1/,0,1792684771468
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=2/,pickup_month=2/,0,1792684771468


21847

In [0]:
# Завдання 4. Групування й агрегація

pickups_by_zip = (
    trips_from_files
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.avg("fare_per_mile").cast("decimal(14,2)").alias("avg_fare_per_mile")
    )
    .orderBy(F.asc("pickup_zip"))

)

dropoffs_by_zip = (
    trips_from_files
    .groupBy("dropoff_zip")
    .agg(
        F.count("*").alias("trips_finished")
    )
    .orderBy(F.asc("dropoff_zip"))
)

display(pickups_by_zip)
display(dropoffs_by_zip)

pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.49
7030,1,40.0,6.35
7087,1,31.0,3.58
7311,1,60.0,30.00
7718,1,5.0,6.25
7737,1,15.5,5.31
10001,1227,13028.01,6.27
10002,611,7439.5,5.43
10003,1180,12963.0,7.61
10005,101,1602.5,4.74


dropoff_zip,trips_finished
7002,2
7008,1
7024,2
7030,4
7087,1
7114,28
7201,1
7302,1
7304,1
7310,1


In [0]:
# Завдання 5. Зʼєднання і значення null

zip_joined = (
    dropoffs_by_zip
    .join(
    pickups_by_zip,
    dropoffs_by_zip.dropoff_zip == pickups_by_zip.pickup_zip, "left"
    )
)

display(zip_joined)


dropoff_zip,trips_finished,pickup_zip,trips_started,total_fare,avg_fare_per_mile
10022,482,10022,517,5090.0,6.21
10002,486,10002,611,7439.5,5.43
10103,472,10103,580,7285.51,6.26
10278,160,10278,167,2169.5,5.69
10280,91,10280,103,1681.5,5.21
10465,3,null,null,null,null
7302,1,null,null,null,null
10710,1,null,null,null,null
11575,1,null,null,null,null
10003,929,10003,1180,12963.0,7.61


In [0]:
# Завдання 5.2. Індекси без посадок

zips_without_pickups = (
    zip_joined
    .filter(
        F.col("pickup_zip").isNull()
        )


)

display(zips_without_pickups)


dropoff_zip,trips_finished,pickup_zip,trips_started,total_fare,avg_fare_per_mile
10465,3,null,null,null,null
7302,1,null,null,null,null
10710,1,null,null,null,null
11575,1,null,null,null,null
7304,1,null,null,null,null
10462,3,null,null,null,null
11415,2,null,null,null,null
10463,9,null,null,null,null
11360,2,null,null,null,null
11010,1,null,null,null,null


In [0]:
# Завдання 5.3. Підсумковий звіт

zip_report = (
    zip_joined
    .select(
        F.col("dropoff_zip").alias("zip"),
        "trips_finished",
        F.coalesce(F.col("trips_started"), F.lit(0)).alias("trips_started"),
        (F.col("trips_finished") - F.coalesce(F.col("trips_started"), F.lit(0))).alias("net_flow"),
        F.coalesce(F.col("total_fare"), F.lit(0)).alias("total_fare"),
        "avg_fare_per_mile"
    )
    .orderBy(F.asc("zip"))
)

display(zip_report)

zip,trips_finished,trips_started,net_flow,total_fare,avg_fare_per_mile
7002,2,2,0,21.0,7.49
7008,1,0,1,0.0,null
7024,2,0,2,0.0,null
7030,4,1,3,40.0,6.35
7087,1,1,0,31.0,3.58
7114,28,0,28,0.0,null
7201,1,0,1,0.0,null
7302,1,0,1,0.0,null
7304,1,0,1,0.0,null
7310,1,0,1,0.0,null


In [0]:
# Завдання 6. Конвеєр одним ланцюжком і запит SQL

pickups_chain = (
    spark
    .table("samples.nyctaxi.trips")
    .select(
        "tpep_pickup_datetime", 
        "trip_distance", 
        "fare_amount", 
        "pickup_zip", 
        "dropoff_zip"
    )
    .filter(
        (F.col("pickup_zip").isNotNull()) 
        & (F.col("dropoff_zip").isNotNull())
        & (F.col("trip_distance") > 0)
        & (F.col("fare_amount") > 0)
    )
    .withColumn(
        "fare_per_mile",
        F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
    )
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.round(F.avg("fare_per_mile"), 2).alias("avg_fare_per_mile")
    )
    .orderBy("pickup_zip")
   
    )


display(pickups_chain)


pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.49
7030,1,40.0,6.35
7087,1,31.0,3.58
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.31
10001,1227,13028.01,6.27
10002,611,7439.5,5.43
10003,1180,12963.0,7.61
10005,101,1602.5,4.74


In [0]:
# Завдання 6. Конвеєр одним ланцюжком і запит SQL

pickups_sql = spark.sql("""
         
WITH pickups AS (
SELECT
    tpep_pickup_datetime,
    pickup_zip, 
    trip_distance, 
    fare_amount,
    dropoff_zip,
    ROUND((fare_amount / trip_distance), 2) AS fare_per_mile
    
FROM samples.nyctaxi.trips
WHERE pickup_zip IS NOT NULL 
    AND dropoff_zip IS NOT NULL 
    AND trip_distance > 0 
    AND fare_amount > 0
)

SELECT 
    pickup_zip,
    COUNT(*) AS trips_started,
    SUM(fare_amount) AS total_fare,
    ROUND(AVG(fare_per_mile), 2) AS avg_fare_per_mile
FROM pickups
GROUP BY pickup_zip
ORDER BY pickup_zip

""")

display(pickups_sql)

pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.49
7030,1,40.0,6.35
7087,1,31.0,3.58
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.31
10001,1227,13028.01,6.27
10002,611,7439.5,5.43
10003,1180,12963.0,7.61
10005,101,1602.5,4.74


In [0]:
# Завдання 7.1 Запис у таблицю

target_zip_report = "workspace.default.zip_report"

zip_report.write.mode("overwrite").saveAsTable(target_zip_report)
print("Table has been created!", target_zip_report)

catalog_zip_report = spark.table("workspace.default.zip_report")

display(catalog_zip_report.orderBy("zip"))

Table has been created! workspace.default.zip_report


zip,trips_finished,trips_started,net_flow,total_fare,avg_fare_per_mile
7002,2,2,0,21.0,7.49
7008,1,0,1,0.0,null
7024,2,0,2,0.0,null
7030,4,1,3,40.0,6.35
7087,1,1,0,31.0,3.58
7114,28,0,28,0.0,null
7201,1,0,1,0.0,null
7302,1,0,1,0.0,null
7304,1,0,1,0.0,null
7310,1,0,1,0.0,null


In [0]:
# Завдання 7.2. Прибирання

spark.sql("""
          DROP TABLE IF EXISTS workspace.default.zip_report;
          """)
print("Table has been dropped!", targer_zip_report)


dbutils.fs.rm(output_path, True)


spark.sql("""
          DROP VOLUME IF EXISTS workspace.default.taxi_files;
          """)
print("Volume has been dropped!", "workspace.default.taxi_files")        

Table has been dropped! workspace.default.zip_report
Volume has been dropped! workspace.default.taxi_files
